In [ ]:
import pandas as pd
import numpy as np

df=pd.read_csv("../data/Finance_Final/model_data_final_ffill.csv")

In [3]:
df_copy=df.copy()

In [4]:
df_copy.dtypes

date                   object
OilPrice              float64
RealInterestRate      float64
CPI                   float64
DollarIndex           float64
VIX                   float64
IndustryProduction    float64
CPE                   float64
OilInventories        float64
OPECProduction        float64
OilProduction         float64
TermSpread            float64
TreasuryYield         float64
FedFundsRate          float64
is_monday               int64
is_friday               int64
is_trading_day          int64
oil_return            float64
oil_return_lag1       float64
oil_return_lag5       float64
oil_volatility_5      float64
inventory_change      float64
dollar_return         float64
vix_change            float64
dtype: object

In [5]:
df_copy.head(10)

,date,OilPrice,RealInterestRate,CPI,DollarIndex,VIX,IndustryProduction,CPE,OilInventories,OPECProduction,...,is_monday,is_friday,is_trading_day,oil_return,oil_return_lag1,oil_return_lag5,oil_volatility_5,inventory_change,dollar_return,vix_change
0,2006-01-01,NaN,2.064842,199.3,NaN,NaN,98.1999,NaN,NaN,33237.829,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2006-01-02,NaN,2.064842,199.3,101.4155,NaN,98.1999,NaN,NaN,33237.829,...,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2006-01-03,63.11,2.064842,199.3,100.7558,11.14,98.1999,NaN,NaN,33237.829,...,0,0,1,NaN,NaN,NaN,NaN,NaN,-0.006505,NaN
3,2006-01-04,63.41,2.064842,199.3,100.2288,11.37,98.1999,NaN,NaN,33237.829,...,0,0,1,0.004754,NaN,NaN,NaN,NaN,-0.005230,0.23
4,2006-01-05,62.81,2.064842,199.3,100.2992,11.31,98.1999,NaN,NaN,33237.829,...,0,0,1,-0.009462,0.004754,NaN,NaN,NaN,0.000702,-0.06
5,2006-01-06,64.21,2.064842,199.3,100.0241,11.00,98.1999,NaN,302584.0,33237.829,...,0,1,1,0.022289,-0.009462,NaN,NaN,NaN,-0.002743,-0.31
6,2006-01-09,63.56,2.064842,199.3,100.1794,11.13,98.1999,NaN,302584.0,33237.829,...,1,0,1,-0.010123,0.022289,NaN,NaN,0.0,0.001553,0.13
7,2006-01-10,63.41,2.064842,199.3,100.1436,10.86,98.1999,NaN,302584.0,33237.829,...,0,0,1,-0.002360,-0.010123,NaN,0.013340,0.0,-0.000357,-0.27
8,2006-01-11,63.91,2.064842,199.3,99.8710,10.94,98.1999,NaN,302584.0,33237.829,...,0,0,1,0.007885,-0.002360,0.004754,0.013629,0.0,-0.002722,0.08
9,2006-01-12,63.96,2.064842,199.3,100.0643,11.20,98.1999,NaN,302584.0,33237.829,...,0,0,1,0.000782,0.007885,-0.009462,0.012241,0.0,0.001935,0.26


In [6]:
df_copy.loc[df["OilPrice"].isna(), "date"]

0       2006-01-01
1       2006-01-02
11      2006-01-16
36      2006-02-20
66      2006-04-01
           ...    
5309    2026-02-01
5320    2026-02-16
5330    2026-03-01
5342    2026-03-17
5343    2026-03-18
Name: date, Length: 275, dtype: object

275개 결측

In [7]:
missing_dates = df_copy.loc[df_copy["OilPrice"].isna(), "date"]

In [8]:
missing_dates

0       2006-01-01
1       2006-01-02
11      2006-01-16
36      2006-02-20
66      2006-04-01
           ...    
5309    2026-02-01
5320    2026-02-16
5330    2026-03-01
5342    2026-03-17
5343    2026-03-18
Name: date, Length: 275, dtype: object

In [9]:
#missing_dates.to_csv("missing_dates.csv", index=False)
#print("\n저장 완료")

In [10]:
import pandas as pd
import numpy as np
from pandas.tseries.holiday import USFederalHolidayCalendar, GoodFriday

# 날짜형으로 변환
md = pd.DataFrame({"date": pd.to_datetime(missing_dates)})

# 날짜만 남기기
md["date"] = md["date"].dt.normalize()

# 분석 범위
start = md["date"].min()
end = md["date"].max()

# 미국 연방공휴일
us_holidays = USFederalHolidayCalendar().holidays(start=start, end=end)

# 성금요일(Good Friday) 추가
good_fridays = GoodFriday.dates(start, end)

# 특수 휴장일 직접 추가
special_closures = pd.to_datetime([
    "2025-01-09",   # Jimmy Carter 국가 애도의 날
])

# 휴일 전후 단축거래/공표 생략 가능성이 큰 날짜들
# (당신 데이터에서 실제 결측으로 나타난 예외 날짜 반영)
adjacent_or_early_close = pd.to_datetime([
    "2006-07-03",
    "2017-07-03",
    "2019-07-05",
    "2006-11-24",
    "2018-11-23",
    "2020-11-27",
    "2021-11-26",
    "2018-12-24",
    "2018-12-31",
])

def classify_missing(d):
    weekday = d.day_name()

    if d.weekday() >= 5:
        return "주말(비거래일)"
    elif d in us_holidays:
        return "미국 연방공휴일"
    elif d in good_fridays:
        return "Good Friday(성금요일)"
    elif d in special_closures:
        return "특별 휴장일"
    elif d in adjacent_or_early_close:
        return "공휴일 인접일/조기종료일 가능성"
    else:
        return "시장 휴장 패턴 아님 → merge/수집 오류 가능성"

md["weekday"] = md["date"].dt.day_name()
md["missing_reason"] = md["date"].apply(classify_missing)

print(md)

           date    weekday                 missing_reason
0    2006-01-01     Sunday                       주말(비거래일)
1    2006-01-02     Monday                       미국 연방공휴일
11   2006-01-16     Monday                       미국 연방공휴일
36   2006-02-20     Monday                       미국 연방공휴일
66   2006-04-01   Saturday                       주말(비거래일)
...         ...        ...                            ...
5309 2026-02-01     Sunday                       주말(비거래일)
5320 2026-02-16     Monday                       미국 연방공휴일
5330 2026-03-01     Sunday                       주말(비거래일)
5342 2026-03-17    Tuesday  시장 휴장 패턴 아님 → merge/수집 오류 가능성
5343 2026-03-18  Wednesday  시장 휴장 패턴 아님 → merge/수집 오류 가능성

[275 rows x 3 columns]


In [11]:
print(md["missing_reason"].value_counts())

missing_reason
미국 연방공휴일                         172
주말(비거래일)                          71
Good Friday(성금요일)                 20
공휴일 인접일/조기종료일 가능성                  9
시장 휴장 패턴 아님 → merge/수집 오류 가능성      2
특별 휴장일                             1
Name: count, dtype: int64


In [12]:
# 제거할 날짜들: 휴장일 + merge/수집 오류 가능성
remove_dates = md.loc[
    md["missing_reason"].isin([
        "주말(비거래일)",
        "미국 연방공휴일",
        "Good Friday(성금요일)",
        "특별 휴장일",
        "공휴일 인접일/조기종료일 가능성",
        "시장 휴장 패턴 아님 → merge/수집 오류 가능성"
    ]),
    "date"
]

# df_copy의 date도 같은 형식으로 맞추기
df_copy["date"] = pd.to_datetime(df_copy["date"]).dt.normalize()

# 해당 날짜 행 제거
df_copy = df_copy[~df_copy["date"].isin(remove_dates)].reset_index(drop=True)

print(df_copy)

           date  OilPrice  RealInterestRate     CPI  DollarIndex    VIX  \
0    2006-01-03     63.11          2.064842  199.30     100.7558  11.14   
1    2006-01-04     63.41          2.064842  199.30     100.2288  11.37   
2    2006-01-05     62.81          2.064842  199.30     100.2992  11.31   
3    2006-01-06     64.21          2.064842  199.30     100.0241  11.00   
4    2006-01-09     63.56          2.064842  199.30     100.1794  11.13   
...         ...       ...               ...     ...          ...    ...   
5064 2026-03-10     83.71          1.618032  327.46     118.7255  24.93   
5065 2026-03-11     86.80          1.618032  327.46     119.2885  24.23   
5066 2026-03-12     95.61          1.618032  327.46     119.8227  27.29   
5067 2026-03-13     98.48          1.618032  327.46     120.5518  27.19   
5068 2026-03-16     93.39          1.618032  327.46          NaN  23.51   

      IndustryProduction      CPE  OilInventories  OPECProduction  ...  \
0                98.1999 

In [13]:
df_copy["OilPrice"].isna().sum()

np.int64(0)

In [14]:
df_copy.columns

Index(['date', 'OilPrice', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX',
       'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction',
       'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate',
       'is_monday', 'is_friday', 'is_trading_day', 'oil_return',
       'oil_return_lag1', 'oil_return_lag5', 'oil_volatility_5',
       'inventory_change', 'dollar_return', 'vix_change'],
      dtype='object')

In [15]:
#임시 ffill()
df_copy["DollarIndex"] = df_copy["DollarIndex"].ffill()
df_copy["TermSpread"] = df_copy["TermSpread"].ffill()
df_copy["TreasuryYield"] = df_copy["TreasuryYield"].ffill()

#원래 기준으로 잡았던 return값
df_copy["oil_return_raw"] = df_copy["OilPrice"].pct_change(fill_method=None)

# t 시점까지 알 수 있는 정보만 사용하도록 한 칸 shift
df_copy["oil_return"] = df_copy["oil_return_raw"]
df_copy["oil_return_shift"] = df_copy["oil_return_raw"].shift(1)

# 하루 전 유가 수익률
df_copy["oil_return_lag1"] = df_copy["oil_return_raw"].shift(1)
df_copy["oil_return_lag1_shift"] = df_copy["oil_return"].shift(2)

# 1주 전 유가 수익률
df_copy["oil_return_lag5"] = df_copy["oil_return_raw"].shift(5)
df_copy["oil_return_lag5_shift"] = df_copy["oil_return"].shift(6)

# 5일 변동성 (t일까지의 변동성을 t+1 예측에 쓰기 위해 shift 1)
df_copy["oil_volatility_5"] = df_copy["oil_return_raw"].rolling(5).std()
df_copy["oil_volatility_5_shift"] = df_copy["oil_return"].rolling(5).std().shift(1)

# 20일 변동성 (정보 누수 방지 위해 shift)
df_copy["oil_volatility_20"] = df_copy["oil_return"].rolling(20).std()
df_copy["oil_volatility_20_shift"] = df_copy["oil_return"].rolling(20).std().shift(1)

# 재고 변화
df_copy["inventory_change"] = df_copy["OilInventories"].diff()

# 달러지수 변화율
df_copy["dollar_return"] = df_copy["DollarIndex"].pct_change(fill_method=None)

# VIX 변화
df_copy["vix_change"] = df_copy["VIX"].diff()

#1 TermSpread 변화량
df_copy["termspread_change"] = df_copy["TermSpread"].diff()

#3 VIX > 30 더미변수 (금융 공포시장 국면)
df_copy["vix_over_30"] = (df_copy["VIX"] > 30).astype(int)

#4 TermSpread < 0 더미변수 (혹시 회귀가 잘 못잡을까봐 추가)
df_copy["termspread_inversion"] = (df_copy["TermSpread"] < 0).astype(int) 


우선, dollarindex, termspread, treasuryYield는 일단 ffill로 채웠습니다. 대게 유가의 분포가 그 전날의 경제지표를 따라가는 느낌이니 ffill로 일단 채워봤습니다. 추후에 수정사항 있을시 수정하겠습니다.

In [16]:
#df_copy.to_csv("trading dates.csv", index=False)
#print("\n저장 완료")

In [23]:
# 5일 이동평균
df_copy["MA_5"] = df_copy["OilPrice"].rolling(5).mean()
df_copy["MA_5_shift"] = df_copy["OilPrice"].rolling(5).mean().shift(1)


# 20일 이동평균
df_copy["MA_20"] = df_copy["OilPrice"].rolling(20).mean()
df_copy["MA_20_shift"] = df_copy["OilPrice"].rolling(20).mean().shift(1)


# MA_5가 MA_20보다 큰지 여부
df_copy["MA_5_gt_MA_20"] = (df_copy["MA_5"] > df_copy["MA_20"]).astype(int)
df_copy["MA_5_gt_MA_20_shift"] = (df_copy["MA_5_shift"] > df_copy["MA_20_shift"]).astype(int)

# TermSpread 
# 이 친구는 전일대비 증가 감소 버전 
# df_copy["TermSpread_direction"] = np.sign(df_copy["TermSpread"].diff()).fillna(0).astype(int)
# TermSpread
# Data가 양수일시 1, 같으면 0 음수일시 -1
df_copy["TermSpread_sign"] = np.sign(df_copy["TermSpread"]) 

# MA 비율, (MA_5 / MA_20), (MA_ratio가 1보다 크면 단기 평균이 장기 평균보다 높음)
df_copy["MA_ratio"] = df_copy["MA_5"] / df_copy["MA_20"]

# OPEC 생산 변화율
df_copy["OPEC_prod_change"] = df_copy["OPECProduction"].pct_change(fill_method=None)
df_copy["OPEC_prod_change"] = df_copy["OPEC_prod_change"].replace(0, pd.NA).ffill()

# 불필요한 원본 수익률 열 제거
df_copy = df_copy.drop(columns=["oil_return_raw"])

C:\Users\ekf44\AppData\Local\Temp\ipykernel_14344\489291408.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_copy["OPEC_prod_change"] = df_copy["OPEC_prod_change"].replace(0, pd.NA).ffill()


KeyError: "['oil_return_raw'] not found in axis"

In [24]:
df_copy.isna().sum()

date                         0
OilPrice                     0
RealInterestRate             0
CPI                          0
DollarIndex                  0
VIX                          4
IndustryProduction           0
CPE                        249
OilInventories               3
OPECProduction               0
OilProduction                3
TermSpread                   0
TreasuryYield                0
FedFundsRate                 0
is_monday                    0
is_friday                    0
is_trading_day               0
oil_return                   1
oil_return_lag1              2
oil_return_lag5              6
oil_volatility_5             5
inventory_change             4
dollar_return                1
vix_change                   8
oil_return_shift             2
oil_return_lag1_shift        3
oil_return_lag5_shift        7
oil_volatility_5_shift       6
oil_volatility_20           20
oil_volatility_20_shift     21
termspread_change            1
vix_over_30                  0
termspre

In [25]:
#df_copy.to_csv("test1.csv", index=False)
#print("\n저장 완료")

In [26]:
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar, GoodFriday

# -----------------------------
# 0. 날짜형 정리
# -----------------------------
df_copy["date"] = pd.to_datetime(df_copy["date"]).dt.normalize()
md["date"] = pd.to_datetime(md["date"]).dt.normalize()

# -----------------------------
# 1. 전체 날짜 범위 만들기
#    df_copy와 md를 모두 반영해서 시작/끝 날짜 설정
# -----------------------------
start_date = min(df_copy["date"].min(), md["date"].min())
end_date = max(df_copy["date"].max(), md["date"].max())

full_dates = pd.DataFrame({
    "date": pd.date_range(start=start_date, end=end_date, freq="D")
})

# -----------------------------
# 2. 전체 날짜에 df_copy 붙이기
# -----------------------------
df_full = full_dates.merge(df_copy, on="date", how="left")

# -----------------------------
# 3. 주말 여부
# -----------------------------
df_full["is_weekend"] = df_full["date"].dt.weekday >= 5

# -----------------------------
# 4. 미국 연방공휴일 / 성금요일 / 특수휴장일
# -----------------------------
start = df_full["date"].min()
end = df_full["date"].max()

us_holidays = USFederalHolidayCalendar().holidays(start=start, end=end)
good_fridays = GoodFriday.dates(start, end)
special_closures = pd.to_datetime(["2025-01-09"])

df_full["is_holiday"] = (
    df_full["date"].isin(us_holidays)
    | df_full["date"].isin(good_fridays)
    | df_full["date"].isin(special_closures)
)

# -----------------------------
# 5. md의 missing_reason 다시 붙이기
# -----------------------------
reason_map = md.set_index("date")["missing_reason"]
df_full["missing_reason"] = df_full["date"].map(reason_map)

# -----------------------------
# 6. md에 없던 빈 날짜들도 이유 자동 채우기
# -----------------------------
def fill_missing_reason(row):
    if pd.notna(row["missing_reason"]):
        return row["missing_reason"]
    if pd.notna(row["OilPrice"]):
        return None
    if row["is_weekend"]:
        return "주말(비거래일)"
    elif row["date"] in us_holidays:
        return "미국 연방공휴일"
    elif row["date"] in good_fridays:
        return "Good Friday(성금요일)"
    elif row["date"] in special_closures:
        return "특별 휴장일"
    else:
        return "기타 결측/비거래 가능성"

df_full["missing_reason"] = df_full.apply(fill_missing_reason, axis=1)

# 확인
print(df_full.loc[df_full["date"].isin(pd.to_datetime(["2006-01-01", "2006-01-02"]))])

        date  OilPrice  RealInterestRate  CPI  DollarIndex  VIX  \
0 2006-01-01       NaN               NaN  NaN          NaN  NaN   
1 2006-01-02       NaN               NaN  NaN          NaN  NaN   

   IndustryProduction  CPE  OilInventories  OPECProduction  ...  MA_20  \
0                 NaN  NaN             NaN             NaN  ...    NaN   
1                 NaN  NaN             NaN             NaN  ...    NaN   

   MA_20_shift  MA_5_gt_MA_20  MA_5_gt_MA_20_shift  TermSpread_sign  MA_ratio  \
0          NaN            NaN                  NaN              NaN       NaN   
1          NaN            NaN                  NaN              NaN       NaN   

   OPEC_prod_change  is_weekend  is_holiday  missing_reason  
0               NaN        True       False        주말(비거래일)  
1               NaN       False        True        미국 연방공휴일  

[2 rows x 45 columns]


In [27]:
df_full.tail(30)

,date,OilPrice,RealInterestRate,CPI,DollarIndex,VIX,IndustryProduction,CPE,OilInventories,OPECProduction,...,MA_20,MA_20_shift,MA_5_gt_MA_20,MA_5_gt_MA_20_shift,TermSpread_sign,MA_ratio,OPEC_prod_change,is_weekend,is_holiday,missing_reason
7352,2026-02-17,62.53,1.748093,327.46,117.7375,20.29,102.551,16700.2,419815.0,35002.24103,...,62.6390,62.4825,1.0,1.0,1.0,1.014256,-0.00377,False,False,None
7353,2026-02-18,65.33,1.748093,327.46,117.8426,19.62,102.551,16700.2,419815.0,35002.24103,...,62.8905,62.6390,1.0,1.0,1.0,1.013794,-0.00377,False,False,None
7354,2026-02-19,66.66,1.748093,327.46,118.2354,20.23,102.551,16700.2,419815.0,35002.24103,...,63.2045,62.8905,1.0,1.0,1.0,1.014643,-0.00377,False,False,None
7355,2026-02-20,66.69,1.748093,327.46,117.9917,19.09,102.551,16700.2,435804.0,35002.24103,...,63.5770,63.2045,1.0,1.0,1.0,1.020054,-0.00377,False,False,None
7356,2026-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,주말(비거래일)
7357,2026-02-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,주말(비거래일)
7358,2026-02-23,66.36,1.748093,327.46,117.9395,21.01,102.551,16700.2,435804.0,35002.24103,...,63.8600,63.5770,1.0,1.0,1.0,1.025900,-0.00377,False,False,None
7359,2026-02-24,65.62,1.748093,327.46,117.9463,19.55,102.551,16700.2,435804.0,35002.24103,...,64.1180,63.8600,1.0,1.0,1.0,1.031411,-0.00377,False,False,None
7360,2026-02-25,65.30,1.748093,327.46,117.7690,17.93,102.551,16700.2,435804.0,35002.24103,...,64.2810,64.1180,1.0,1.0,1.0,1.028702,-0.00377,False,False,None
7361,2026-02-26,65.10,1.748093,327.46,117.9042,18.63,102.551,16700.2,435804.0,35002.24103,...,64.3985,64.2810,1.0,1.0,1.0,1.021980,-0.00377,False,False,None


In [28]:
df_full["is_weekend"] = df_full["is_weekend"].astype(int)
df_full["is_holiday"] = df_full["is_holiday"].astype(int)

In [29]:
df_full["missing_reason"].value_counts(dropna=False)

missing_reason
None                             5069
주말(비거래일)                         2109
미국 연방공휴일                          172
Good Friday(성금요일)                  20
공휴일 인접일/조기종료일 가능성                   9
시장 휴장 패턴 아님 → merge/수집 오류 가능성       2
특별 휴장일                              1
Name: count, dtype: int64

In [30]:
df_full["is_weekend"].value_counts()

is_weekend
0    5273
1    2109
Name: count, dtype: int64

In [31]:
df_full["is_holiday"].value_counts()

is_holiday
0    7153
1     229
Name: count, dtype: int64

In [32]:
df_full["is_non_trading_day"] = ((df_full["is_weekend"] == 1) | (df_full["is_holiday"] == 1)).astype(int)

In [33]:
df_full = df_full.drop(columns=["is_weekend", "is_holiday"])

In [34]:
cols = list(df_full.columns)

i = cols.index("missing_reason")
j = cols.index("is_non_trading_day")

cols[i], cols[j] = cols[j], cols[i]

df_full = df_full[cols]

In [35]:
df_full.columns

Index(['date', 'OilPrice', 'RealInterestRate', 'CPI', 'DollarIndex', 'VIX',
       'IndustryProduction', 'CPE', 'OilInventories', 'OPECProduction',
       'OilProduction', 'TermSpread', 'TreasuryYield', 'FedFundsRate',
       'is_monday', 'is_friday', 'is_trading_day', 'oil_return',
       'oil_return_lag1', 'oil_return_lag5', 'oil_volatility_5',
       'inventory_change', 'dollar_return', 'vix_change', 'oil_return_shift',
       'oil_return_lag1_shift', 'oil_return_lag5_shift',
       'oil_volatility_5_shift', 'oil_volatility_20',
       'oil_volatility_20_shift', 'termspread_change', 'vix_over_30',
       'termspread_inversion', 'MA_5', 'MA_5_shift', 'MA_20', 'MA_20_shift',
       'MA_5_gt_MA_20', 'MA_5_gt_MA_20_shift', 'TermSpread_sign', 'MA_ratio',
       'OPEC_prod_change', 'is_non_trading_day', 'missing_reason'],
      dtype='object')

In [36]:
df_full = df_full[[
    # 원본 데이터
    "date",
    "OilPrice",
    "RealInterestRate",
    "CPI",
    "DollarIndex",
    "VIX",
    "IndustryProduction",
    "CPE",
    "OilInventories",
    "OPECProduction",
    "OilProduction",
    "TermSpread",
    "TreasuryYield",
    "FedFundsRate",

    # 유가 수익률 / 변동성
    "oil_return",
    "oil_return_shift",
    "oil_return_lag1",
    "oil_return_lag1_shift",
    "oil_return_lag5",
    "oil_return_lag5_shift",
    "oil_volatility_5",
    "oil_volatility_5_shift",
    "oil_volatility_20",
    "oil_volatility_20_shift",

    # 이동평균 관련
    "MA_5",
    "MA_5_shift",
    "MA_20",
    "MA_20_shift",
    "MA_ratio",
    "MA_5_gt_MA_20",
    "MA_5_gt_MA_20_shift",

    # 거시 / 금융 파생 변수
    "dollar_return",
    "vix_change",
    "vix_over_30",
    "inventory_change",
    "termspread_change",
    "termspread_inversion",
    "TermSpread_sign",
    "OPEC_prod_change",

    # 날짜 / 거래일 관련
    "is_monday",
    "is_friday",
    "is_trading_day",
    "is_non_trading_day",
    "missing_reason",
]]

In [ ]:
df_full.to_csv("../data/Finance_Final/Cleaning data_fix.csv", index=False)
print("\n저장 완료")